In [ ]:
# libraries
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/cleaned/mobile_reviews_cleaned.csv', parse_dates=['review_date'])

In [3]:
print(df.shape)
print(df.dtypes)
df.head()

(50000, 20)
review_date             datetime64[us]
verified_purchase                 bool
brand                              str
model                              str
country                            str
language                           str
currency                           str
source                             str
sentiment                          str
age                              int64
price_usd                      float64
price_local                    float64
exchange_rate_to_usd           float64
rating                         float64
battery_life_rating              int64
camera_rating                    int64
performance_rating               int64
design_rating                    int64
display_rating                   int64
helpful_votes                    int64
dtype: object


,review_date,verified_purchase,brand,model,country,language,currency,source,sentiment,age,price_usd,price_local,exchange_rate_to_usd,rating,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,helpful_votes
0,2023-11-06,True,Realme,Realme 12 Pro,India,Hindi,INR,Amazon,Negative,45,337.31,27996.73,83.00,2.0,1,1,3,2,1,1
1,2023-03-30,True,Realme,Realme 12 Pro,Brazil,Portuguese,BRL,Flipkart,Positive,18,307.78,1754.35,5.70,4.0,3,2,4,3,2,5
2,2022-12-07,True,Google,Pixel 6,India,Hindi,INR,Aliexpress,Positive,27,864.53,71755.99,83.00,4.0,3,5,3,2,4,8
3,2025-03-11,False,Xiaomi,Redmi Note 13,Uae,English,AED,Amazon,Positive,19,660.94,2425.65,3.67,3.0,1,3,2,1,2,3
4,2023-09-29,True,Motorola,Edge 50,Brazil,Portuguese,BRL,Bestbuy,Neutral,38,792.13,4515.14,5.70,3.0,3,3,2,2,1,0


In [4]:
product_df = (df.groupby(['brand', 'model'])
                .agg(reviews=('rating', 'size'),
                     avg_price=('price_usd', 'mean'),
                     avg_rating=('rating', 'mean'),
                     battery=('battery_life_rating', 'mean'),
                     camera=('camera_rating', 'mean'),
                     performance=('performance_rating', 'mean'),
                     design=('design_rating', 'mean'),
                     display=('display_rating', 'mean'),
                     helpful_votes=('helpful_votes', 'mean'),
                     positive_pct=('sentiment', lambda s: (s == 'Positive').mean() * 100))
                .round(2)
                .reset_index())

print(product_df.shape)
product_df

(22, 12)


,brand,model,reviews,avg_price,avg_rating,battery,camera,performance,design,display,helpful_votes,positive_pct
0,Apple,Iphone 13,1767,1108.27,3.13,2.75,2.75,2.77,2.77,2.71,3.74,57.39
1,Apple,Iphone 14,1788,1099.22,3.15,2.74,2.78,2.72,2.77,2.76,3.69,57.72
2,Apple,Iphone 15 Pro,1793,1103.41,3.06,2.68,2.71,2.66,2.68,2.69,3.57,53.65
3,Apple,Iphone Se,1796,1103.38,3.08,2.73,2.70,2.73,2.69,2.74,3.58,54.90
4,Google,Pixel 6,2379,807.92,3.09,2.71,2.71,2.69,2.71,2.72,3.69,56.41
5,Google,Pixel 7A,2468,806.82,3.11,2.72,2.73,2.73,2.72,2.75,3.62,55.51
6,Google,Pixel 8,2387,798.97,3.10,2.73,2.72,2.69,2.74,2.74,3.60,55.63
7,Motorola,Edge 50,2315,508.70,3.07,2.67,2.70,2.70,2.69,2.71,3.63,54.56
8,Motorola,Moto G Power,2351,504.61,3.12,2.72,2.71,2.72,2.72,2.72,3.65,56.32
9,Motorola,Razr 40,2395,506.15,3.10,2.71,2.68,2.73,2.72,2.72,3.63,53.65


In [5]:
product_df.to_csv('../data/processed/product_level.csv', index=False)

In [6]:
spec_cols = ['battery_life_rating', 'camera_rating', 'performance_rating',
             'design_rating', 'display_rating']

df['avg_spec_score'] = df[spec_cols].mean(axis=1).round(2)   # overall spec score
df['verified_purchase'] = df['verified_purchase'].astype(int)  # True/False -> 1/0

df[['avg_spec_score', 'verified_purchase']].describe().T

,count,mean,std,min,25%,50%,75%,max
avg_spec_score,50000.0,2.7181,1.099568,1.0,1.8,2.8,3.6,5.0
verified_purchase,50000.0,0.8014,0.398950,0.0,1.0,1.0,1.0,1.0


In [7]:
cat_cols = ['brand', 'country', 'sentiment', 'source']

df_encoded = pd.get_dummies(df, columns=cat_cols, dtype=int)

print(df_encoded.shape)
df_encoded.filter(regex='^(brand|country|sentiment|source)_').head()

(50000, 41)


,brand_Apple,brand_Google,brand_Motorola,brand_Oneplus,brand_Realme,brand_Samsung,brand_Xiaomi,country_Australia,country_Brazil,country_Canada,...,country_Usa,sentiment_Negative,sentiment_Neutral,sentiment_Positive,source_Aliexpress,source_Amazon,source_Bestbuy,source_Ebay,source_Flipkart,source_Unknown
0,0,0,0,0,1,0,0,0,0,0,...,0,1,0,0,0,1,0,0,0,0
1,0,0,0,0,1,0,0,0,1,0,...,0,0,0,1,0,0,0,0,1,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,1,1,0,0,0,0,0
3,0,0,0,0,0,0,1,0,0,0,...,0,0,0,1,0,1,0,0,0,0
4,0,0,1,0,0,0,0,0,1,0,...,0,0,1,0,0,0,1,0,0,0


In [8]:
from sklearn.preprocessing import StandardScaler

features = ['price_usd', 'avg_spec_score', 'rating', 'helpful_votes', 'age']

X = df_encoded[features]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)

print(X_scaled.describe().T[['mean', 'std']].round(2))

                mean  std
price_usd       -0.0  1.0
avg_spec_score  -0.0  1.0
rating          -0.0  1.0
helpful_votes    0.0  1.0
age             -0.0  1.0


In [9]:
import joblib

In [10]:
df_encoded.to_csv('../data/processed/reviews_encoded.csv', index=False)
X_scaled.to_csv('../data/processed/reviews_scaled.csv', index=False)

In [11]:
joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']